# Rerun Only the Failures

In this video, I'm going to compute a color for each of 1,673 photos with Pixeltable. We'll keep the results that worked and rerun only the photos that failed.

In a real job, computing that color is a model call, and you don't want to pay for every photo again because a few failed.

We heard about this from two people. One asked whether a run that stops partway loses the whole batch, and another called rerunning only the failed photos "massive."

First, we'll stop a run partway and pick it back up. Then we'll fix four kinds of failure, one at a time.

## Start from an empty `rerun_demo` directory

Start Jupyter with `uv run jupyter lab` in this folder. Nothing here calls a paid API, so you don't need any keys.

In [ ]:
!uv sync

In [ ]:
import pixeltable as pxt
import pixeltable.functions as pxtf

# Start fresh: drop the tables from any earlier run.
pxt.drop_dir('rerun_demo', force=True, if_not_exists='ignore')

## The photos table computes two columns and an embedding from each photo

In `photos`, the only columns you fill in are `name` and `image`. `avg_color` and `color` are computed columns: Pixeltable computes `avg_color` from each image, then `color` from `avg_color`, on every insert.

The last line adds an embedding index on `image`, so Pixeltable embeds each photo on insert too.

In [ ]:
!sed -n '5,17p' schema.py

`average_color` stands in for a model call, and `image_embedding` stands in for an embedding model. They're the demo-only part: each one times out the first time it sees about 1 photo in 50, the way a busy API does.

In [ ]:
!sed -n '11,41p' udfs.py

`pxt schema diff` shows the plan, and `pxt schema update` creates the table.

In [ ]:
!pxt schema diff schema.py rerun_demo

In [ ]:
!pxt schema update schema.py rerun_demo -f

## By default, an insert saves all of its rows or none

`photo_names.txt` lists 1,673 small ImageNet photos from the Pixeltable repo. Each `image` is a URL, and Pixeltable downloads the photo when it inserts the row. I'll insert the first 200 the default way.

In [ ]:
BASE = 'https://raw.githubusercontent.com/pixeltable/pixeltable/main/tests/data/imagenette2-160/'
names = open('photo_names.txt').read().split()
rows = [{'name': name, 'image': BASE + name} for name in names]

photos = pxt.get_table('rerun_demo.photos')
try:
    photos.insert(rows[:200])
except pxt.Error as e:
    print(e)
photos.count()

One photo hit a TypeError, so the insert saved none of the 200. That's the default, `on_error='abort'`, and it means a table never holds half of an insert.

## Chunks keep what finished when the run stops

For a big run, we change two things. The first is `on_error='ignore'`, which keeps going past a failed photo and saves its error on the row. The second is inserting 200 photos at a time, so Pixeltable saves each chunk as soon as it finishes.

I'll stop this cell partway through, the way a crash or a closed laptop would.

In [ ]:
def insert_remaining(rows, chunk_size=200):
    done = {row['name'] for row in photos.select(photos.name).collect()}
    todo = [row for row in rows if row['name'] not in done]
    for start in range(0, len(todo), chunk_size):
        photos.insert(todo[start : start + chunk_size], on_error='ignore')


insert_remaining(rows)

*Stop the cell after three or four chunks.*

Pixeltable kept the chunks that finished. Only the chunk in progress is gone.

In [ ]:
photos.count()

I'll run it again. `insert_remaining` skips the photos already in the table, so it picks up where it stopped.

In [ ]:
insert_remaining(rows)
photos.count()

All 1,673 photos are in. Here are a few that worked. The `color` column holds red, green, or blue: whichever channel is highest in `avg_color`.

In [ ]:
photos.where(photos.avg_color.errortype == None).select(
    photos.name, photos.image, photos.avg_color, photos.color
).limit(4).collect()

## Each color that failed stays on its row, with its error

A photo that failed is still a row. `avg_color.errortype` holds the exception's name, and you can group and filter on it.

In [ ]:
photos.group_by(photos.avg_color.errortype).select(
    photos.avg_color.errortype, photos=pxtf.count(photos.name)
).collect()

`pxt errors` lists the same failures from the command line, with each message. `color` failed on the same photos, because it's built on `avg_color`. The `next:` lines give the command that retries them.

In [ ]:
!pxt errors rerun_demo/photos 2>&1 | tail -4

## A retry fixes the timeouts

Timeouts are the failures a retry can fix. In Python, `errors_only=True` reruns `avg_color` only on the rows where it failed, then the columns built on it.

In [ ]:
photos.recompute_columns('avg_color', errors_only=True).num_rows

That's how many rows it reran: only the failures, not all 1,673. The stand-in times out only once, so the timeouts are gone now.

In [ ]:
photos.group_by(photos.avg_color.errortype).select(
    photos.avg_color.errortype, photos=pxtf.count(photos.name)
).collect()

The TypeErrors failed again, because a retry runs the same code on the same photos.

## A code fix, then a retry, covers the grayscale photos

The TypeError means `average_color` expected three color channels. Mode `L` means grayscale, with one channel, and 28 photos are like that.

In [ ]:
photos.where(photos.avg_color.errortype == 'TypeError').select(
    photos.name, photos.image, photos.image.mode, photos.avg_color.errormsg
).limit(3).collect()

The fix is one line in the schema: convert each photo to RGB before `average_color` runs.

In [ ]:
!diff schema.py schema_rgb.py

In [ ]:
!pxt schema diff schema_rgb.py rerun_demo

In [ ]:
!pxt schema update schema_rgb.py rerun_demo -f

`pxt schema update` keeps the colors we already computed. Converting an RGB photo to RGB doesn't change it, so there's no reason to compute those again.

This time I'm going to use the CLI to recompute `avg_color`. `--errors-only` reruns only the rows that failed, like `errors_only=True` in Python.

In [ ]:
!pxt recompute rerun_demo/photos avg_color --errors-only -f

It reran 29 rows: the 28 grayscale photos, and one file that failed again. Back in Python, I'll reload `photos` first, because its schema changed.

In [ ]:
photos = pxt.get_table('rerun_demo.photos')  # reload after the schema change
photos.group_by(photos.avg_color.errortype).select(
    photos.avg_color.errortype, photos=pxtf.count(photos.name)
).collect()

The grayscale photos have colors now.

## A file that isn't an image gets deleted

`bad_image.JPEG` is 29 bytes of text with a `.JPEG` name. Pixeltable couldn't load it as an image, so `image` holds the error, and `average_color` never ran.

In [ ]:
photos.where(photos.avg_color.errortype != None).select(
    photos.name, photos.image.errortype, photos.avg_color.errortype
).collect()

A retry or a code fix won't help here, so we delete the row.

In [ ]:
photos.delete(where=photos.name == 'bad_image.JPEG')

photos.count(), photos.where(photos.avg_color.errortype != None).count()

## An update re-embeds only the photos missing an embedding

The embedding index timed out too, on a different 1 in 50 photos. Those failures don't show up in `pxt errors`, and there's no command to recompute an index by name.

To find them, I'll look for photos whose embedding is empty.

In [ ]:
missing = photos.image.embedding(idx='photo_embedding') == None
photos.where(missing).count()

There's a trick that works: update `image` to its own value, only on those photos. It looks odd, but it makes the index embed them again. `cascade=False` leaves `avg_color` and `color` as they are, so `average_color` doesn't run again.

In [ ]:
photos.update({'image': photos.image}, where=missing, cascade=False)
photos.where(missing).count()

Every photo has an embedding now.

## The table's history shows every step

Let's wrap up with the history of `photos`. Each change is a version.

In [ ]:
!pxt history rerun_demo/photos

The list starts at the bottom, with version 0. Every chunk of the insert got its own version, and the default insert that failed left none.

## Each kind of failure got one fix

We ran 1,673 photos. When the run stopped partway, the chunks that finished stayed saved, and the next run picked up from there. Then each kind of failure got its own fix.

| Failure | Photos | Fix |
|---|---|---|
| Timeout, from the stand-in | about 1 in 50 | a retry: `recompute_columns(..., errors_only=True)` |
| Grayscale photo | 28 | a code fix, then a retry: `pxt recompute ... --errors-only` |
| Not an image | 1 | delete the row |
| Embedding timeout, from the stand-in | about 1 in 50 | `update` the column to its own value where the embedding is missing |

The notebook and schema files are in the `rerun-failures` folder of the repo linked below. If you liked this video, please consider starring the Pixeltable GitHub repository. Stars help other developers find the project.